# Pipeline Sanity Check — Mock Env (no game needed)

Runs the full actor-learner cycle end-to-end on Colab using `MockGameEnv`: collect synthetic rollouts -> serialize to `.npz` -> PPO gradient update -> reload checkpoint -> collect again. If this passes, the only untested pieces are capture/detection on the live game.

In [ ]:
!git clone https://github.com/YOUR_USER/RL_gaming_agent.git repo 2>/dev/null || (cd repo && git pull)
%cd repo
!pip install -q -r requirements-train.txt

In [ ]:
# Round 1: collect with a fresh policy on the mock env
from src.agent.collect import collect_rollouts
from src.agent.model import make_ppo
from src.config import load_config
from src.env.mock_game_env import MockGameEnv

config = load_config()
env = MockGameEnv()
model = make_ppo(env, config)
model.save('checkpoints/best.zip')
collect_rollouts(env, model, n_steps=512, output_path='rollouts/rollouts.npz')

In [ ]:
# Learner step: gradient update on the serialized rollouts
from src.agent.train_colab import train_on_rollouts

small = {**config, 'ppo': {**config['ppo'], 'n_steps': 512}}
model = train_on_rollouts(
    rollouts_path='rollouts/rollouts.npz',
    checkpoint_path='checkpoints/best.zip',
    output_path='checkpoints/best.zip',
    config=small,
)

In [ ]:
# Round 2: reload the updated checkpoint and collect again (closes the loop)
from src.agent.model import load_or_init

model = load_or_init('checkpoints/best.zip', env, config)
collect_rollouts(env, model, n_steps=256, output_path='rollouts/rollouts_round2.npz')
print('Full actor-learner cycle OK')

In [ ]:
# Evaluate on the mock env
from src.agent.evaluate import evaluate

evaluate(env, model, episodes=3)